In [10]:
#load the recording
from pathlib import Path
import pandas as pd

project = Path(r"C:\Users\redhu\OneDrive\Desktop\Bearing_fault_detection")
registry = pd.read_csv(project / "metadata/recording_registry.csv")

development = registry[
    registry["partition"].eq("development")
    & registry["eligible_for_analysis"].eq(True)
].copy()

print("Selected recordings:", len(development))

Selected recordings: 272


In [11]:
#set the window length.
fs = 64_000
window_seconds = 1
window_samples = int(fs * window_seconds)

print("Samples per window:", window_samples)

Samples per window: 64000


In [12]:
#oad one recording
from scipy.io import loadmat

row = development.iloc[0]
path = project / "data/raw/paderborn" / row["relative_path"]
recording = loadmat(path)[path.stem][0, 0]

channels = recording["Y"].ravel()
names = [channel["Name"].item() for channel in channels]
vibration = channels[names.index("vibration_1")]
x = vibration["Data"].ravel().astype(float)

print("Recording:", row["recording_id"])
print("Vibration samples:", x.size)

Recording: N09_M07_F10_K001_2
Vibration samples: 256001


In [13]:
n_windows = x.size // window_samples
windows = x[:n_windows * window_samples].reshape(n_windows, window_samples)

print("Windows shape:", windows.shape)
print("Unused samples:", x.size % window_samples)

Windows shape: (4, 64000)
Unused samples: 1


In [14]:
#remove the mean from each window
windows = windows - windows.mean(axis=1, keepdims=True)

print("First window mean:", windows[0].mean())

First window mean: 2.1094237467877974e-18


In [15]:
window_info = pd.DataFrame({
    "recording_id": row["recording_id"],
    "bearing_id": row["bearing_id"],
    "condition": row["condition"],
    "window_no": range(n_windows)
})
window_info["start_s"] = window_info["window_no"] * window_seconds
window_info["end_s"] = window_info["start_s"] + window_seconds

window_info

,recording_id,bearing_id,condition,window_no,start_s,end_s
0,N09_M07_F10_K001_2,K001,healthy,0,0,1
1,N09_M07_F10_K001_2,K001,healthy,1,1,2
2,N09_M07_F10_K001_2,K001,healthy,2,2,3
3,N09_M07_F10_K001_2,K001,healthy,3,3,4


In [16]:
import numpy as np

assert np.isfinite(windows).all(), "Invalid vibration values."
assert np.allclose(windows.mean(axis=1), 0, atol=1e-10), "Check mean removal."
assert len(window_info) == len(windows), "Window metadata count differs."

print("Window checks passed.")

Window checks passed.


In [17]:
def make_windows(path):
    recording = loadmat(path)[path.stem][0, 0]
    channels = recording["Y"].ravel()
    names = [channel["Name"].item() for channel in channels]
    vibration = channels[names.index("vibration_1")]
    x = vibration["Data"].ravel().astype(float)

    n = x.size // window_samples
    windows = x[:n * window_samples].reshape(n, window_samples)
    return windows - windows.mean(axis=1, keepdims=True)

In [18]:
windows = make_windows(path)
print(windows.shape)

(4, 64000)


In [19]:
#create and save windows for every eligible development recording
window_folder = project / "data/interim/windows"
window_folder.mkdir(parents=True, exist_ok=True)

counts = []
for _, row in development.iterrows():
    path = project / "data/raw/paderborn" / row["relative_path"]
    windows = make_windows(path)
    assert windows.size > 0 and np.isfinite(windows).all(), row["recording_id"]
    np.save(window_folder / f"{row['recording_id']}.npy", windows)
    counts.append({"recording_id": row["recording_id"], "n_windows": len(windows)})

window_counts = pd.DataFrame(counts)
print("Recordings processed:", len(window_counts))
print("Total windows:", window_counts["n_windows"].sum())

Recordings processed: 272
Total windows: 1085


In [20]:
records = development.merge(window_counts, on="recording_id", validate="one_to_one")
window_registry = records.loc[
    records.index.repeat(records["n_windows"])
].reset_index(drop=True)

window_registry["window_no"] = window_registry.groupby("recording_id").cumcount()
window_registry["start_s"] = window_registry["window_no"] * window_seconds
window_registry["end_s"] = window_registry["start_s"] + window_seconds
window_registry["window_file"] = (
    "data/interim/windows/" + window_registry["recording_id"] + ".npy"
)

window_registry[["recording_id", "window_no", "start_s", "end_s"]].head()

,recording_id,window_no,start_s,end_s
0,N09_M07_F10_K001_2,0,0,1
1,N09_M07_F10_K001_2,1,1,2
2,N09_M07_F10_K001_2,2,2,3
3,N09_M07_F10_K001_2,3,3,4
4,N09_M07_F10_K001_3,0,0,1


In [21]:
assert len(window_registry) == window_counts["n_windows"].sum()
assert not window_registry.duplicated(["recording_id", "window_no"]).any()
assert window_registry["partition"].eq("development").all()

output = project / "metadata/window_registry.csv"
window_registry.to_csv(output, index=False)

print("Saved windows:", len(window_registry))
print("Saved to:", output)

Saved windows: 1085
Saved to: C:\Users\redhu\OneDrive\Desktop\Bearing_fault_detection\metadata\window_registry.csv
